In [8]:
retriever_tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

clause_matrix = retriever_tfidf.fit_transform(
    clauses_df["text"]
)

print("Clause TF-IDF shape:", clause_matrix.shape)

Clause TF-IDF shape: (47321, 30000)


In [9]:
def retrieve_clauses(document_id, hypothesis, top_k=5):
    
    document_id = str(document_id)
    
    doc_indices = clauses_df.index[
        clauses_df["document_id"] == document_id
    ].tolist()
    
    if not doc_indices:
        return pd.DataFrame()
    
    hypothesis_vector = retriever_tfidf.transform(
        [hypothesis]
    )
    
    scores = cosine_similarity(
        hypothesis_vector,
        clause_matrix[doc_indices]
    ).flatten()
    
    ranked_positions = np.argsort(scores)[::-1][:top_k]
    
    selected_indices = [
        doc_indices[i]
        for i in ranked_positions
    ]
    
    result = clauses_df.loc[selected_indices].copy()
    
    result["retrieval_score"] = scores[ranked_positions]
    
    return result

In [10]:
def create_retrieved_context(row, top_k=5):
    
    retrieved = retrieve_clauses(
        row["document_id"],
        row["hypothesis"],
        top_k=top_k
    )
    
    if retrieved.empty:
        return ""
    
    return " ".join(
        retrieved["text"].tolist()
    )


for split_name in ["train", "dev", "test"]:
    
    mask = ml_df["split"] == split_name
    
    ml_df.loc[mask, "retrieved_context"] = (
        ml_df.loc[mask]
        .apply(create_retrieved_context, axis=1)
    )

print("Retrieved contexts created.")

Retrieved contexts created.


In [11]:
ml_df["bert_text"] = (
    ml_df["hypothesis"].fillna("")
    + " [SEP] "
    + ml_df["retrieved_context"].fillna("")
)

display(
    ml_df[
        [
            "split",
            "hypothesis",
            "retrieved_context",
            "choice"
        ]
    ].head(3)
)

,split,hypothesis,retrieved_context,choice
0,train,Receiving Party shall not reverse engineer any...,The Recipient shall not be precluded from disc...,NotMentioned
1,train,Receiving Party shall destroy or return some C...,Either Party may terminate the working relatio...,Entailment
2,train,Agreement shall not grant Receiving Party any ...,4. Nothing in this Agreement is to be construe...,Entailment


In [12]:
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

print("Legal-BERT loaded.")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

Legal-BERT loaded.


In [13]:
train_df = ml_df[ml_df["split"] == "train"].copy()
dev_df   = ml_df[ml_df["split"] == "dev"].copy()
test_df  = ml_df[ml_df["split"] == "test"].copy()

X_train = train_df["bert_text"].tolist()
y_train = train_df["target"].values

X_dev = dev_df["bert_text"].tolist()
y_dev = dev_df["target"].values

X_test = test_df["bert_text"].tolist()
y_test = test_df["target"].values

print("Train:", len(X_train))
print("Dev:", len(X_dev))
print("Test:", len(X_test))

Train: 7191
Dev: 1037
Test: 2091


In [14]:
MAX_LENGTH = 512

train_encodings = tokenizer(
    X_train,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH
)

dev_encodings = tokenizer(
    X_dev,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH
)

test_encodings = tokenizer(
    X_test,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH
)

print(
    "Train token shape:",
    (len(train_encodings["input_ids"]),
     len(train_encodings["input_ids"][0]))
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Train token shape: (7191, 512)


In [15]:
class ContractNLIDataset(Dataset):
    
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(
            labels,
            dtype=torch.long
        )
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        
        item = {
            key: torch.tensor(value[idx])
            for key, value in self.encodings.items()
        }
        
        item["labels"] = self.labels[idx]
        
        return item


train_dataset = ContractNLIDataset(
    train_encodings,
    y_train
)

dev_dataset = ContractNLIDataset(
    dev_encodings,
    y_dev
)

test_dataset = ContractNLIDataset(
    test_encodings,
    y_test
)

print(len(train_dataset))
print(len(dev_dataset))
print(len(test_dataset))

7191
1037
2091


In [16]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Device:", device)
print("Train batches:", len(train_loader))

Device: cuda
Train batches: 450


In [17]:
classes = np.array([0, 1, 2])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
).to(device)

print("Class weights:")

for class_id, weight in zip(classes, class_weights):
    print(
        class_id,
        round(weight.item(), 4)
    )

Class weights:
0 0.85
1 0.679
2 2.8502


In [18]:
optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

loss_fn = torch.nn.CrossEntropyLoss(
    weight=class_weights
)

print("Optimizer and loss ready.")

Optimizer and loss ready.


In [19]:
EPOCHS = 2

for epoch in range(EPOCHS):
    
    model.train()
    
    total_loss = 0
    
    for step, batch in enumerate(train_loader):
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        loss = loss_fn(
            outputs.logits,
            labels
        )
        
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )
        
        optimizer.step()
        
        total_loss += loss.item()
        
        if (step + 1) % 100 == 0:
            print(
                f"Epoch {epoch+1}/{EPOCHS} "
                f"| Step {step+1}/{len(train_loader)} "
                f"| Loss {loss.item():.4f}"
            )
    
    avg_loss = total_loss / len(train_loader)
    
    print(
        f"\nEpoch {epoch+1} completed "
        f"| Average loss: {avg_loss:.4f}\n"
    )

KeyboardInterrupt: 

In [ ]:
model.eval()

dev_predictions = []
dev_actuals = []

with torch.no_grad():
    
    for batch in dev_loader:
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )
        
        dev_predictions.extend(
            predictions.cpu().numpy()
        )
        
        dev_actuals.extend(
            labels.cpu().numpy()
        )


print(
    classification_report(
        dev_actuals,
        dev_predictions,
        target_names=[
            "NotMentioned",
            "Entailment",
            "Contradiction"
        ],
        digits=4
    )
)

print(
    "Balanced Accuracy:",
    round(
        balanced_accuracy_score(
            dev_actuals,
            dev_predictions
        ),
        4
    )
)

In [ ]:
model.eval()

test_predictions = []
test_actuals = []

with torch.no_grad():
    
    for batch in test_loader:
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )
        
        test_predictions.extend(
            predictions.cpu().numpy()
        )
        
        test_actuals.extend(
            labels.cpu().numpy()
        )


print(
    classification_report(
        test_actuals,
        test_predictions,
        target_names=[
            "NotMentioned",
            "Entailment",
            "Contradiction"
        ],
        digits=4
    )
)

In [ ]:
MODEL_SAVE_PATH = "/kaggle/working/legalbert_contractnli"

model.save_pretrained(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

print("Model saved to:", MODEL_SAVE_PATH)

In [20]:
import os

print(os.path.exists("/kaggle/working/legalbert_contractnli"))
print(os.listdir("/kaggle/working"))

False
['.virtual_documents']
